# 03 — Silver Layer Transformation & Data Quality: Real Estate (Property Listings, Market Valuation & AVM Pricing)

**Purpose:** Clean, deduplicate, type-cast, enrich, and enable Delta Change Data Feed (CDF) on `real_estate_prod.silver.silver_properties_clean` and `real_estate_prod.silver.silver_sales_enriched`.

In [ ]:
# Databricks Widget Parameters (overridden by Lakeflow Jobs / Databricks Asset Bundles)
dbutils.widgets.text("catalog_name", "real_estate_prod", "Unity Catalog Name")
CATALOG = dbutils.widgets.get("catalog_name")
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_landing"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/bronze/checkpoints"
print(f"Active Catalog: {CATALOG} | Landing Volume: {VOLUME_PATH}")

## Step 1: Execute Silver Cleansing, Enrichment & Delta CDF Enablement

In [ ]:
from pyspark.sql import functions as F

bronze_props = spark.table(f"{CATALOG}.bronze.bronze_property_listings")
silver_props = (
    bronze_props
    .dropDuplicates(["property_id"])
    .filter((F.col("sqft") >= 300) & (F.col("list_price_usd") > 25000))
    .withColumn("listing_date", F.to_date("listing_date"))
    .withColumn("price_per_sqft", F.round(F.col("list_price_usd") / F.col("sqft"), 2))
    .withColumn("property_age_years", F.lit(2026) - F.col("year_built"))
    .drop("_rescued_data")
)
silver_props.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.silver_properties_clean")
spark.sql(f"ALTER TABLE {CATALOG}.silver.silver_properties_clean SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

bronze_sales = spark.table(f"{CATALOG}.bronze.bronze_property_sales")
silver_sales = (
    bronze_sales
    .dropDuplicates(["sale_id"])
    .withColumn("sale_to_list_ratio", F.col("sale_to_list_ratio").cast("double"))
    .withColumn("close_date", F.to_date("close_date"))
    .join(silver_props.select("property_id", "city", "property_type", "sqft", "list_price_usd"), on="property_id", how="inner")
    .withColumn("closed_price_usd", F.round(F.col("list_price_usd") * F.col("sale_to_list_ratio"), 0))
    .drop("_rescued_data")
)
silver_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.silver_sales_enriched")
print("Silver Real Estate tables created successfully.")

## Step 2: Optimize Silver Tables (Liquid Clustering / Z-Order & Table Stats)

In [ ]:
for tbl in ["silver_properties_clean", "silver_sales_enriched"]:
    spark.sql(f"OPTIMIZE {CATALOG}.silver.{tbl}")
    spark.sql(f"ANALYZE TABLE {CATALOG}.silver.{tbl} COMPUTE STATISTICS")
print("Silver Delta tables optimized and statistics refreshed.")